# P4 — Files, the course's data and keeping your work

**Python primer · session 4**

## What this session is for

Everything so far lived inside the notebook. Real work does not. Data arrives in files and **your own work has to survive the day**.

So the session has three parts:

- First **files**, in four formats:
  - text
  - CSV
  - JSON
  - HDF5

  HDF5 is the format that scientific data usually comes in.
- Then the course's own data: a file of strain from a detector and a table of the events found so far.
- Last, how to trust a notebook and keep it safe with git.

The marks and the temperatures are made up. The numbers about GW150914 and GW170817 come from the two discovery papers (LIGO Scientific Collaboration and Virgo Collaboration 2016) and (LIGO Scientific Collaboration and Virgo Collaboration 2017). The strain file and the catalogue come from GWOSC, the Gravitational Wave Open Science Center, at <https://gwosc.org/eventapi/html/GWTC-1-confident/GW150914/> and <https://gwosc.org/eventapi/csv/allevents/>, as of October 2026.

## Text files

A **text file** is lines of characters and nothing else. Use it for a log or a list of names. A class register has one name on each line. The call `open('register.txt', 'w')` opens a file. The second argument is the mode: `'w'` is for writing and `'r'` is for reading. The words `with ... as f:` give the open file the name `f` and close it when the indented block ends. The call `f.write(text)` puts the text in the file and adds nothing to it. The two characters `\n` stand for the end of a line, so `f.write(name + '\n')` writes the name and starts a new line. The sign `+` joins two strings. A `for` loop over a file gives one line at a time, with its `\n` still at the end. The call `line.strip()` removes that and any spaces at the two ends.

In [ ]:
names = ["Anna", "Bruno", "Chiara", "Dario", "Elisa"]

with open('register.txt', 'w') as f:       # 'w' deletes what was there
    for name in names:
        f.write(name + '\n')               # \n ends the line

with open('register.txt', 'r') as f:       # 'r' reads: the default
    for line in f:
        print(line.strip())               # strip removes the \n

> **Tip — Always open a file with a with block**
>
> The `with` block opens the file and closes it when the block ends, **even if something inside fails**. Always use it.

> **Careful — A missing file usually means the wrong folder**
>
> If Python says the file is not there, you are most often not in the folder you think. `import os` and then `os.getcwd()` tells you where you are.

## Modules and imports

A **module** is a Python file you can `import`. A **library** is a collection of modules written by somebody else. Five libraries carry this course:

- `numpy` for arrays
- `matplotlib` for plots
- `pandas` for tables
- `h5py` for HDF5 files
- `scipy` for numerical methods

The line `import math` opens the module and `math.sqrt(16)` reaches the function `sqrt` inside it with a dot. The line `from math import sqrt, pi` brings chosen names in, so you write `sqrt` alone. The line `import numpy as np` gives the library a short name.

In [ ]:
import math
print(math.sqrt(16), math.pi)         # the name in front: clearest

from math import sqrt, pi             # bring chosen names in
print(sqrt(16), pi)

import numpy as np                    # a conventional short name
print(np.sqrt(16))

> **Tip — Write np, pd and plt the usual way**
>
> Everybody writes `np`, `pd` and `plt` the same way and so should you.

An `ImportError` means the library is not installed. The course environment already has the five.

## CSV with pandas

A **CSV** file is a table as plain text, with one row on each line and commas between the columns. CSV means comma separated values. Here are the marks of six students in three exams. The marks are made up. Write them as text. A string between three quotes, `"""`, may run over several lines. The next cell keeps the whole table in one such string and writes it to a file with `f.write`.

In [ ]:
rows = """name,maths,physics,chemistry
Anna,27,30,24
Bruno,21,18,25
Chiara,30,28,29
Dario,18,22,19
Elisa,25,27,30
Franco,19,24,21
"""

with open('marks.csv', 'w') as f:
    f.write(rows)

The first line is the **header**: it names the columns. The library `pandas` reads the file into a **DataFrame**, a table with named columns and a row index. The call `pd.read_csv('marks.csv')` reads the file and gives back a DataFrame. The attribute `table.shape` is the number of rows and columns. A cell that ends with a name shows its value, here the whole table.

In [ ]:
import pandas as pd

table = pd.read_csv('marks.csv')
print(table.shape)                       # rows, columns
table

The **`describe`** method gives, for each column:

- the count
- the mean
- the spread
- the minimum
- the quartiles
- the maximum

The spread is the standard deviation. The quartiles cut the sorted values into four equal parts.

In the code, `table[exams]` keeps the columns named in the list `exams`. Then `.describe()` gives the summary and `.round(2)` rounds every number to two decimals.

In [ ]:
exams = ['maths', 'physics', 'chemistry']
table[exams].describe().round(2)

In [ ]:
print(table[exams].mean().round(2))     # the mean of each column

A **selection** is the boolean mask of P3 again. `table['mean'] >= 24` is `True` or `False` for each row and `table[mask]` keeps the rows that are `True`. `axis=1` means across the columns of each row, which gives one mean per student. The line `table['mean'] = ...` creates a new column with that name. In `good[['name', 'mean']]` the inner brackets make a list of column names and the outer brackets select those columns. The call `len(good)` counts the rows.

In [ ]:
table['mean'] = table[exams].mean(axis=1)   # one mean per student
good = table[table['mean'] >= 24]            # keep the rows that are True

print(good[['name', 'mean']].round(2))
print(len(good), "of", len(table))

A DataFrame **draws itself**. `plot.bar` makes one group of bars for each row and `set_index('name')` puts the names on the horizontal axis. The call gives back the axes, so you can label them. The argument `figsize=(8, 3.5)` sets the width and the height of the figure in inches. The call `ax.set_title(...)` writes a title.

In [ ]:
import matplotlib.pyplot as plt

by_name = table.set_index('name')[exams]
ax = by_name.plot.bar(figsize=(8, 3.5))
ax.set_xlabel('student')
ax.set_ylabel('mark')
ax.set_title('Three exams for six students (made up)')
plt.show()

The same table draws other plots. `plot.box` draws one box for each exam. The box covers the middle half of the marks and the line inside it is the middle mark. The second plot needs `stack()`, which puts the three columns into one long column of 18 marks. A **histogram** counts how many values fall in each bin, which is a range of values. The method `hist` draws it.

In [ ]:
ax = table[exams].plot.box(figsize=(6, 3))
ax.set_ylabel('mark')
ax.set_title('The spread of each exam (made up)')
plt.show()

In [ ]:
marks_all = table[exams].stack()          # one column of 18 marks
ax = marks_all.hist(bins=6, figsize=(6, 3))
ax.set_xlabel('mark')
ax.set_ylabel('number of marks')
ax.set_title('All 18 marks (made up)')
plt.show()

## JSON: nested data

**JSON** is text that holds a nested structure of names, lists and values. You reach for it for a record, a configuration or anything a web service hands you. The call `json.dump(record, f, indent=2)` writes the record into the open file `f` and `indent=2` lays it out in readable lines. The call `json.load(f)` reads it back. The record is a dictionary that holds a list and another dictionary. The expression `back["marks"]["physics"]` takes two steps: first the inner dictionary and then its key.

In [ ]:
import json

record = {"name": "Anna", "courses": ["maths", "physics"],
          "marks": {"maths": 27, "physics": 30}}

with open('student.json', 'w') as f:
    json.dump(record, f, indent=2)
with open('student.json', 'r') as f:
    back = json.load(f)

print(back["marks"]["physics"], back["courses"])

An object becomes a dictionary and a list stays a list. The record is made up.

## HDF5 with h5py

A text file with a million numbers would be slow and enormous. Scientific data is stored in binary and the usual format is **HDF5**. An HDF5 file is a small file system inside one file:

- **Groups** are like folders.
- **Datasets** are arrays.
- **Attributes** are small pieces of metadata attached to either.

Make one. It holds a week of hourly temperatures with the number of readings per hour stored beside them. The temperatures are made up: a smooth rise and fall every 24 hours plus a small random scatter. In the code, `h5py.File('example.h5', 'w')` opens the file for writing as `open` does for text. The call `f.create_group('week')` makes a group. The call `g.create_dataset('temperatures', data=temps)` makes a dataset inside it and stores the array. The line `d.attrs['units'] = ...` attaches an attribute to the dataset. The formula `18 + 6 * np.sin(...)` is a sine wave with mean 18 degrees. It swings 6 degrees up and down once every 24 hours. The call `rng.normal(0, 0.8, hours.size)` adds the scatter.

In [ ]:
import h5py

per_hour = 1                                   # readings per hour
hours = np.arange(7 * 24 * per_hour) / per_hour    # hour of the week
rng = np.random.default_rng(4)
daily = 18 + 6 * np.sin(2 * np.pi * (hours - 9) / 24)
temps = daily + rng.normal(0, 0.8, hours.size)

with h5py.File('example.h5', 'w') as f:
    g = f.create_group('week')
    d = g.create_dataset('temperatures', data=temps)
    d.attrs['units'] = 'degrees Celsius'
    d.attrs['readings_per_hour'] = per_hour
print('written')

Now open it again and look at what is inside. Print the dataset itself first and then read it with `[:]`. The call `f.keys()` gives the names inside a file or a group and `list(...)` turns them into a list. A path such as `'week/temperatures'` walks through the groups with a slash. The `attrs` of a dataset work like a dictionary, so `.items()` gives each name with its value. The brackets `[:]` are a slice with no start and no stop: they take everything.

In [ ]:
with h5py.File('example.h5', 'r') as f:
    print(list(f.keys()))
    print(list(f['week'].keys()))
    print(f['week/temperatures'])          # a reference to the dataset
    data = f['week/temperatures'][:]       # the numbers, in memory
    print(data.shape)
    for key, value in f['week/temperatures'].attrs.items():
        print(key, value)

> **Important — The colon that reads the data**
>
> `f['week/temperatures']` gives you a *reference* to the dataset, not the numbers.
> `f['week/temperatures'][:]` reads them into memory as an array.
>
> This is why HDF5 handles files larger than your memory. You can look at the
> shape and the attributes first and then read only what you need.

Now take the number of readings per hour from the file and build the time axis from it.

In [ ]:
with h5py.File('example.h5', 'r') as f:
    temps = f['week/temperatures'][:]
    per_hour = f['week/temperatures'].attrs['readings_per_hour']

hours = np.arange(temps.size) / per_hour      # hour of each reading
print(f"values     {temps.size}")
print(f"per hour   {per_hour}")
print(f"duration   {temps.size / per_hour / 24:.1f} days")
print(f"mean       {temps.mean():.1f} degrees")

> **Careful — Data you did not write deserves suspicion**
>
> The number of readings per hour came from the file and the time axis was built from it. Data you did not write always deserves that suspicion.

In [ ]:
days = hours / 24                # the time axis, in days

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(days, temps)
ax.set_xlabel('time (days)')
ax.set_ylabel('temperature (degrees Celsius)')
ax.set_title('A week of hourly temperatures (made up)')
plt.show()

A week is seven days and each day has one warm afternoon. Count the peaks: seven. The file, the code and the plot agree. Any Series or DataFrame has a `.plot` method. It uses Matplotlib underneath, so `plt.xlabel` and the rest still apply. A **Series** is one column of a table: values with an index. Here is the same week in a `Series` with the day as its index.

In [ ]:
week = pd.Series(temps, index=days)       # the day is the index
week.plot(figsize=(8, 3))                 # a Series draws itself too
plt.xlabel('time (days)')
plt.ylabel('temperature (degrees Celsius)')
plt.title('The same week, drawn from a Series (made up)')
plt.show()

## SciPy: the rest of the stack

**`scipy`** is the library of numerical methods built on NumPy:

- fitting
- integration
- statistics
- signal tools
- constants

We use one tool, a straight line through points. Here are forty yearly mean temperatures of a city. They are invented: a slow rise plus a scatter from one year to the next. The call `stats.linregress(x, y)` finds the straight line that fits the points best. It gives back an object whose `.slope` and `.intercept` hold the two numbers of the line. The line `from scipy import stats` takes the part of scipy called `stats`.

In [ ]:
from scipy import stats

years = np.arange(1985, 2025)                 # forty years
since = years - 1985                          # years since the first one
rng = np.random.default_rng(11)
yearly = 14.0 + 0.03 * since + rng.normal(0, 0.4, years.size)

line = stats.linregress(since, yearly)
print(f"slope      {line.slope:.3f} degrees per year")
print(f"intercept  {line.intercept:.2f} degrees")

The **slope** is how many degrees the line rises each year. The **intercept** is its value in the first year. The numbers are invented, so this shows a tool and says nothing about any real city.

## The course's data: the strain file

Lecture 0 gave you the numbers of two black holes that merged and of the detectors that recorded them. Now you open the data itself.

The file `gw150914_H1.hdf5` holds 32 seconds of strain from LIGO Hanford around GW150914. The **strain** is the measurement of a detector, as lecture 0 defined it. The file comes from GWOSC, <https://gwosc.org/eventapi/html/GWTC-1-confident/GW150914/>, as of October 2026.

Open it as you opened `example.h5`. First look at the groups. The path `'../labs/data/...'` starts with `..`, which means one folder up. The name `STRAIN_FILE` is in capitals to say that it is a constant: a value the code never changes.

In [ ]:
STRAIN_FILE = '../labs/data/gw150914_H1.hdf5'

with h5py.File(STRAIN_FILE, 'r') as f:
    print(list(f.keys()))
    print(list(f['strain'].keys()))
    print(f['strain/Strain'])

The file has three groups. The group `strain` holds the measurement, one dataset called `Strain`. The group `meta` holds a few facts about it. The third group is not needed here.

Read the values and the facts. The brackets `[:]` read an array and `[()]` reads a single value. `Xspacing` is the time between two neighbouring values, in seconds. The call `.decode()` turns text stored as bytes into ordinary text. A line that starts with **`assert`** is a check: `assert condition` does nothing when the condition is `True` and stops with an error when it is `False`.

In [ ]:
with h5py.File(STRAIN_FILE, 'r') as f:
    strain = f['strain/Strain'][:]
    spacing_s = f['strain/Strain'].attrs['Xspacing']
    file_s = f['meta/Duration'][()]
    detector = f['meta/Detector'][()].decode()

print(detector, strain.size, "values")
print(file_s, "seconds")
print(round(1 / spacing_s), "values per second")
assert np.isclose(strain.size * spacing_s, file_s)

The file holds 32 seconds at 4096 values per second from LIGO Hanford, whose code is `H1`. The `assert` checks that the numbers agree: the number of values times the spacing is the duration.

The file has no column of times. Build the time axis from the spacing, as you built it from the readings per hour. It starts at 0 s, the start of the file. The argument `linewidth=0.4` draws a thin line, which suits many points.

In [ ]:
t_s = np.arange(strain.size) * spacing_s     # seconds since the file starts
print(t_s[0], t_s[-1])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_s, strain, linewidth=0.4)
ax.set_xlabel('time since the start of the file (s)')
ax.set_ylabel('strain (no unit)')
ax.set_title('32 s of LIGO Hanford around GW150914')
plt.show()

GW150914 sits 15.4 s into the file, from GWOSC. Make a mask for the two seconds around it, with `&` as in P3. Then plot those. The call `ax.axvline(x, ...)` draws a vertical line at `x` and `linestyle='--'` makes it dashed. The expression `t_s[window]` keeps the times where the mask is `True`.

In [ ]:
EVENT_S = 15.4                  # GW150914, seconds into the file
window = (t_s > EVENT_S - 1) & (t_s < EVENT_S + 1)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_s[window], strain[window], linewidth=0.5)
ax.axvline(EVENT_S, color='tab:red', linestyle='--', label='GW150914')
ax.set_xlabel('time since the start of the file (s)')
ax.set_ylabel('strain (no unit)')
ax.legend()
plt.show()

The signal is not visible. What you see is the noise. Lecture 0 said why: the signal sits below the noise (LIGO Scientific Collaboration and Virgo Collaboration 2016).

## The course's data: the catalogue

The file `gwtc_events.csv` is a table of the events found so far, one row for each. It comes from GWOSC, <https://gwosc.org/eventapi/csv/allevents/>, as of October 2026. Read it with pandas, as you read `marks.csv`.

The table has many columns. Keep four: the name, the two masses in solar masses and the distance. A **solar mass** is the mass of the Sun. `luminosity_distance` is the distance, in megaparsecs. A **megaparsec** (Mpc) is a distance used for the sky. **`usecols`** reads only the columns you list and **`rename`** gives them names with the unit. The dictionary `names` maps each old column name to its new one. It replaces the list of students used earlier. The `.rename(columns=names)` call is written straight after `read_csv` because `read_csv` gives back a DataFrame. The call `events.head(3)` shows the first three rows.

In [ ]:
CATALOGUE = '../labs/data/gwtc_events.csv'
keep = ['commonName', 'mass_1_source', 'mass_2_source', 'luminosity_distance']
names = {'commonName': 'name', 'mass_1_source': 'mass_1_sun',
         'mass_2_source': 'mass_2_sun', 'luminosity_distance': 'distance_mpc'}

events = pd.read_csv(CATALOGUE, usecols=keep).rename(columns=names)
print(events.shape)
events.head(3)

Each row is one event. `mass_1_sun` is the heavier mass of the two. Count the events and the rows that hold masses. The call `.count()` counts the values that are present. The call `.isna()` is `True` where a value is missing and `.sum()` counts those, as in P3. A missing value is **`NaN`**, which means *not a number*: the cell in the file is empty.

In [ ]:
print(len(events), "events")
print(events['mass_1_sun'].count(), "have masses")
print(events['mass_1_sun'].isna().sum(), "do not")

The catalogue holds 391 events. Some rows have no masses. The cell is empty and pandas shows `NaN`. Pandas leaves `NaN` out of a count and a histogram and puts it last when it sorts.
**`sort_values`** puts the table in order. The call `sort_values('mass_1_sun')` sorts by that column from the smallest up. The argument `ascending=False` turns it round, so the biggest comes first. The call `.head(1)` keeps the first row. The call `.to_string(index=False)` prints the table as plain text without the row numbers. Which event is the heaviest and which is the farthest?

In [ ]:
heaviest = events.sort_values('mass_1_sun', ascending=False)
farthest = events.sort_values('distance_mpc', ascending=False)
print(heaviest.head(1).to_string(index=False))
print(farthest.head(1).to_string(index=False))

A selection keeps the rows where a condition is `True`. Which events have both masses below three solar masses? Each condition goes in its own brackets and `&` joins them, as in P3.

In [ ]:
both_light = (events['mass_1_sun'] < 3) & (events['mass_2_sun'] < 3)
print(events[both_light].to_string(index=False))

These are the two mergers of neutron stars. One of them is GW170817 (LIGO Scientific Collaboration and Virgo Collaboration 2017).

The same kind of selection finds GW150914 by name.

In [ ]:
first = events[events['name'] == 'GW150914']
print(first.to_string(index=False))

The catalogue numbers for GW150914 differ a little from those announced in 2016, which were 36 and 29 solar masses at 410 Mpc (LIGO Scientific Collaboration and Virgo Collaboration 2016). They are estimates and they have been refined since.

A histogram shows the heavier masses of all the events at once. A histogram counts how many values fall in each bin, which is a range of values.

In [ ]:
n_masses = events['mass_1_sun'].count()
ax = events['mass_1_sun'].hist(bins=20, figsize=(8, 3))
ax.set_xlabel('heavier mass (solar masses)')
ax.set_ylabel('number of events')
ax.set_title(f'The heavier mass of {n_masses} events')
plt.show()

The heavier masses pile up in two places, near 10 and near 40 solar masses. A few events are far heavier. The two neutron-star mergers sit at the far left.

## A notebook you can trust

A notebook can lie. The **kernel** is the Python process that runs your cells. It remembers every name you ever defined, even after you delete the cell that defined it. A notebook can work only because of a variable you deleted an hour ago. It can also work only because you ran the cells in an order you forgot.

> **Important — Restart the kernel and run all**
>
> Before you trust a notebook or send it to anyone, restart the kernel and run all
> the cells from the top. In JupyterLab use *Kernel* and then *Restart Kernel and
> Run All Cells*. In VS Code use **Restart** and then **Run All**.
>
> If a cell fails, the notebook was not telling the truth. Do it now with this
> notebook.

A second habit is the check. An `assert` keeps a check that you would otherwise do by eye and throw away. Here is `average` from P2 with one check.

In [ ]:
def average(marks):
    """Return the mean of a list of marks."""
    return sum(marks) / len(marks)


assert np.isclose(average([18, 24, 30]), 24.0)
print("the check passed")

`assert condition` does nothing when the condition is `True`. It stops with an `AssertionError` when the condition is `False`. **Silence means that the check passed.** The text after a comma is the message you read when it fails.

In [ ]:
try:
    assert average([18.1, 18.4, 18.7]) == 18.4, "the average should be 18.4"
except AssertionError as error:
    print("check failed:", error)

This is why P1 said never to compare floats with `==`. The check passes with `np.isclose`.

## Git, the minimum

P0 said that a deleted codespace takes its files with it.

**Git** keeps a history of your files and a copy outside your codespace. A **commit** is a snapshot of your files, with a message and a time. You can return to any commit. A commit that you **push** to GitHub is a copy that stays.

Git holds your work in **three places**. A change moves from one to the next on purpose.

| Place | What is in it | How a change gets there |
|---|---|---|
| working copy | the files as you edit them | you save the file |
| staging area | the changes chosen for the next commit | `git add` |
| history | the commits | `git commit` |

Type the commands in the terminal of the codespace, in the folder of your notebooks.

```bash
git status                                 # which files changed
git add P3_data_structures_numpy.ipynb     # choose a file for the next commit
git commit -m "Finish the P3 exercises"    # make the snapshot
git push                                   # send your commits to your fork
```

The message after `-m` is a few words on what changed.

> **Tip — Check where you are with git status**
>
> Run `git status` often: it tells you where you are.

Your codespace came from the course repository. After your first push it follows your **fork**, which is your own copy of the course repository on GitHub. `upstream` is the course repository itself.

When the course publishes new notebooks, your own edits can stand in the way. Set them aside, pull and bring them back:

```bash
git stash                  # your edits, set aside
git pull                   # the new notebooks
git stash pop              # your edits, back
```

If you have already made a commit, the second line is `git pull upstream main`. **Commit your work before every pull.**

## Exercises

### 1. Write, read, check

Here is the register of a second class. Write the names to a text file called `register_b.txt`, in alphabetical order, one per line. Read them back and check that the list you read is the sorted list you wrote.

In [ ]:
names_b = ["Giulia", "Franco", "Irene", "Hugo", "Elisa"]

# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 2. The loudest moment of the file

Use `strain` and `t_s` from the strain file. Find the largest absolute strain and the time at which it happens. Print both. Then print whether that time lies within one second of `EVENT_S`.

**What to look for.** Is the loudest moment where the event is?

**Hint.** The call `np.abs(x)` gives each value without its sign and `np.argmax(x)` gives the position of the largest value. In an f-string the code `.1e` writes a number in scientific notation with one decimal.

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


### 3. The nearest events

Use the table `events`. Print the name and the distance of the three nearest events. Then count the events whose heavier mass is above 50 solar masses.

This is the bonus: a sort, a mask and a count.

In [ ]:
# your code here

> **Solution**
>
> Typed out together in the session. Write your own version first — the error you
> make on the way is the part you keep.


## Where this leaves you

You can write and read text files and tables and HDF5 files and plot what is in them. You have opened the course's own strain file and catalogue. You can check a notebook and keep your work in git.

### References

**References**

- LIGO Scientific Collaboration and Virgo Collaboration (2016). *Observation of Gravitational Waves from a Binary Black Hole Merger*.
- LIGO Scientific Collaboration and Virgo Collaboration (2017). *GW170817: Observation of Gravitational Waves from a Binary Neutron Star Inspiral*.